# Male vs Female Image Classifier Training Pipeline

This notebook loads, preprocesses, and trains a `LinearRegression` model to perform binary classification on gender images. 

### Architecture Constraints & Coding Style:
- **OpenCV** is used for image reading, resizing ($64 \times 64$), and grayscaling.
- **NumPy** and **Pandas** handle features/labels data structure.
- **scikit-learn's LinearRegression** model is the ONLY model used.
- Prediction threshold: $\text{prediction} \ge 0.5 \rightarrow \text{Male (1)}$, and $\text{prediction} < 0.5 \rightarrow \text{Female (0)}$.
- The trained model is serialized using **pickle** as `gender_model.pkl`.

### Multi-threading Optimization:
- Uses a Python standard library `ThreadPoolExecutor` to speed up the process of loading and preprocessing 58,000+ images from disk in parallel.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import cv2
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, r2_score

## 1. Image Preprocessing Pipeline
We preprocess each image using OpenCV:
1. Load from file using `cv2.imread()`.
2. Resize to uniform $64 \times 64$ dimensions.
3. Convert to grayscale.
4. Flatten to a 1D vector of shape $(4096,)$.
5. Scale pixel values to $[0, 1]$ via normalization.

In [ ]:
IMAGE_SIZE = (64, 64)

def preprocess_image(image_path, target_size=IMAGE_SIZE):
    """
    Reads an image using OpenCV, resizes it, grayscales it,
    flattens it, and normalizes the pixel values.
    """
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Unable to read image at: {image_path}")
    
    img_resized = cv2.resize(img, target_size)
    img_gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
    img_flattened = img_gray.flatten()
    
    # Normalize features to [0, 1] for stable regression weights
    img_normalized = img_flattened.astype(np.float32) / 255.0
    return img_normalized

## 2. Load Dataset from Directories in Parallel
We traverse the `dataset/` directory to load images from `male` and `female` categories, mapping them to targets `1` and `0` respectively. We use multi-threading to speed up reading.

In [ ]:
DATASET_DIR = "dataset"

def load_dataset(dataset_dir, target_size=IMAGE_SIZE):
    """
    Loads all images from category folders and returns feature matrix X and labels vector y.
    """
    # Mapping folders to labels (Male = 1, Female = 0)
    category_map = {
        'male': 1,
        'female': 0,
        'Male': 1,
        'Female': 0
    }
    
    print(f"Scanning directory: {dataset_dir}")
    if not os.path.exists(dataset_dir):
        raise FileNotFoundError(f"Dataset path '{dataset_dir}' does not exist.")
        
    tasks = []
    for category in os.listdir(dataset_dir):
        category_path = os.path.join(dataset_dir, category)
        
        if os.path.isdir(category_path) and category in category_map:
            label = category_map[category]
            image_names = os.listdir(category_path)
            for img_name in image_names:
                if img_name.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp')):
                    img_path = os.path.join(category_path, img_name)
                    tasks.append((img_path, label))
                    
    total_images = len(tasks)
    print(f"Found {total_images} total images. Loading and preprocessing in parallel...")
    
    features = []
    labels = []
    
    def process_single_task(task):
        img_path, label = task
        try:
            feat = preprocess_image(img_path, target_size)
            return feat, label
        except Exception:
            return None
            
    success_count = 0
    with ThreadPoolExecutor(max_workers=8) as executor:
        results = executor.map(process_single_task, tasks)
        
        for idx, result in enumerate(results):
            if result is not None:
                features.append(result[0])
                labels.append(result[1])
                success_count += 1
            if (idx + 1) % 5000 == 0:
                print(f"Processed {idx + 1}/{total_images} images...")
                
    print(f"Successfully loaded {success_count} images out of {total_images} total images.")
    return np.array(features), np.array(labels)

## 3. Load Data and Check Balance
Let's call the loader function and review our data properties.

In [ ]:
X, y = load_dataset(DATASET_DIR)

print(f"\nTotal loaded samples: {X.shape[0]}")
print(f"Single image flat dimensions: {X.shape[1]}")

# Print label distributions using Pandas
df_y = pd.Series(y)
print("\nClass distribution:")
print(df_y.value_counts().rename({1: "Male (1)", 0: "Female (0)"}))

## 4. Train-Test Split & Model Fitting
We use `train_test_split` with `random_state=42` and `test_size=0.2`.
We then fit a scikit-learn `LinearRegression` model.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size:  {X_test.shape[0]}")

# Instantiate and fit Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)
print("Linear Regression model successfully trained!")

## 5. Model Evaluation
Evaluate the continuous regression predictions using $R^2$, and convert continuous predictions to classes via $0.5$ threshold to check classification accuracy.

In [ ]:
# Predict continuous variables
y_train_pred_cont = model.predict(X_train)
y_test_pred_cont = model.predict(X_test)

# Convert to binary label outputs using >= 0.5 threshold
y_train_pred_class = (y_train_pred_cont >= 0.5).astype(int)
y_test_pred_class = (y_test_pred_cont >= 0.5).astype(int)

# Scores
train_r2 = r2_score(y_train, y_train_pred_cont)
test_r2 = r2_score(y_test, y_test_pred_cont)

train_acc = accuracy_score(y_train, y_train_pred_class)
test_acc = accuracy_score(y_test, y_test_pred_class)

print("="*50)
print(f"Training R² score:                    {train_r2:.4f}")
print(f"Testing R² score:                     {test_r2:.4f}")
print(f"Training Classification Accuracy:     {train_acc * 100:.2f}%")
print(f"Testing Classification Accuracy:      {test_acc * 100:.2f}%")
print("="*50)

## 6. Save Serialized Model Weight to Pickle File
We dump the trained model state to `gender_model.pkl`.

In [ ]:
MODEL_FILENAME = "gender_model.pkl"

with open(MODEL_FILENAME, "wb") as f:
    pickle.dump(model, f)
print(f"Successfully saved trained model state to '{MODEL_FILENAME}'!")